# DataGateway Test Notebook

End-to-end exercises covering **all major DataGateway features**:

| # | Feature |
|---|---|
| 1 | Environment & seed DB |
| 2 | Structured mode (no table) |
| 3 | Configured mode – field_map translation |
| 4 | Configured mode – no field_map (semantic DB) |
| 5 | DataFrameParams: fieldnames, column_names, index_col |
| 6 | DataFrameOptions: sort, dedup, groupby |
| 7 | sticky_filters & runtime kwargs |
| 8 | exclude=True / use_exclude |
| 9 | persist=True |
| 10 | timeout= |
| 11 | load_period / aload_period |
| 12 | _has_any_rows() |
| 13 | Chunked \_\_in list splitting |
| 14 | from_config() classmethod |


## 1. Environment Setup

In [1]:
import sys, datetime
sys.path.insert(0, "../src")

import pandas as pd
import dask.dataframe as dd
from sqlalchemy import String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from pydantic import SecretStr

from boti_data.db.sql_config import SqlDatabaseConfig
from boti_data.gateway import DataGateway
from boti_data.field_map import FieldMap
from boti_data.gateway import DataFrameParams, DataFrameOptions

print("Imports OK")


Imports OK


## 2. Seed In-Memory Databases

Two SQLite databases:
- **legacy_db** – non-semantic column names (field_map required)
- **semantic_db** – semantic column names (no field_map)

In [2]:
# ---- schema definitions ------------------------------------------------
class LegacyBase(DeclarativeBase): pass
class SemanticBase(DeclarativeBase): pass

class LegacyProduct(LegacyBase):
    __tablename__ = "products"
    id: Mapped[int] = mapped_column(primary_key=True)
    id_tipo_produto: Mapped[int] = mapped_column()
    id_track_global: Mapped[int] = mapped_column()
    codigo_barra: Mapped[str] = mapped_column(String(32))
    event_dt: Mapped[datetime.date] = mapped_column()

class SemanticProduct(SemanticBase):
    __tablename__ = "products"
    id: Mapped[int] = mapped_column(primary_key=True)
    product_type_id: Mapped[int] = mapped_column()
    global_track_id: Mapped[int] = mapped_column()
    barcode: Mapped[str] = mapped_column(String(32))
    event_dt: Mapped[datetime.date] = mapped_column()

ROWS = [
    (1, 10, "A001", datetime.date(2024, 1, 10)),
    (1, 20, "B002", datetime.date(2024, 2, 15)),
    (2, 30, "C003", datetime.date(2024, 3, 20)),
    (2, 40, "D004", datetime.date(2024, 4, 5)),
]

import tempfile, pathlib

# Use a temp directory so relative-path issues can't arise regardless of
# where the kernel's CWD happens to be.
_TMP_DIR = pathlib.Path(tempfile.mkdtemp(prefix="boti_nb_"))
LEGACY_DB  = _TMP_DIR / "legacy_test.db"
SEMANTIC_DB = _TMP_DIR / "semantic_test.db"

LEGACY_DSN   = f"sqlite:///{LEGACY_DB}"
SEMANTIC_DSN = f"sqlite:///{SEMANTIC_DB}"

FIELD_MAP = {
    "id_tipo_produto": "product_type_id",
    "id_track_global": "global_track_id",
    "codigo_barra": "barcode",
    "event_dt": "event_dt",  # same name, no rename
}

def legacy_cfg(query_only=False):
    return SqlDatabaseConfig(connection_url=SecretStr(LEGACY_DSN), query_only=query_only)

def semantic_cfg(query_only=False):
    return SqlDatabaseConfig(connection_url=SecretStr(SEMANTIC_DSN), query_only=query_only)

# ---- create & seed -------------------------------------------------------
legacy_engine = create_engine(LEGACY_DSN)
LegacyBase.metadata.drop_all(legacy_engine)
LegacyBase.metadata.create_all(legacy_engine)
with Session(legacy_engine) as s:
    s.add_all([LegacyProduct(id_tipo_produto=t, id_track_global=g, codigo_barra=b, event_dt=d) for t,g,b,d in ROWS])
    s.commit()

semantic_engine = create_engine(SEMANTIC_DSN)
SemanticBase.metadata.drop_all(semantic_engine)
SemanticBase.metadata.create_all(semantic_engine)
with Session(semantic_engine) as s:
    s.add_all([SemanticProduct(product_type_id=t, global_track_id=g, barcode=b, event_dt=d) for t,g,b,d in ROWS])
    s.commit()

print("Databases seeded.")


Databases seeded.


## 3. Structured Mode (no )

In structured mode you pass , , , etc. directly — no configured-mode translation.

In [3]:
from sqlalchemy import select

gw = DataGateway(legacy_cfg())
try:
    stmt = select(LegacyProduct).where(LegacyProduct.id_tipo_produto == 1)
    df = gw.load(
        statement=stmt,
        model=LegacyProduct,
        as_pandas=True,
    )
    display(df)
    assert len(df) == 2, f"Expected 2 rows, got {len(df)}"
    print("\u2713 Structured mode OK")
finally:
    gw.close()


,id,id_tipo_produto,id_track_global,codigo_barra,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00


✓ Structured mode OK


## 4. Configured Mode — field_map translation

Filters and column names are expressed as **semantic** names; the gateway translates to DB names before querying and renames the result back.

In [4]:
gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    df = gw.load(product_type_id=1, as_pandas=True)
    display(df)
    assert "product_type_id" in df.columns, "DB legacy name leaked into output"
    assert "id_tipo_produto" not in df.columns
    assert len(df) == 2
    print("✓ field_map translation OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00


✓ field_map translation OK


## 5. Configured Mode — No field_map (semantic DB)

When the DB already uses semantic column names no translation is needed.

In [5]:
gw = DataGateway(semantic_cfg(), table="products")
try:
    df = gw.load(product_type_id=2, as_pandas=True)
    display(df)
    assert len(df) == 2
    print("✓ No field_map passthrough OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,3,2,30,C003,2024-03-20 00:00:00+00:00
1,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ No field_map passthrough OK


## 6. DataFrameParams: fieldnames, column_names, index_col

In [6]:
# fieldnames: only return a subset of columns
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    df_params=DataFrameParams(fieldnames=("global_track_id", "barcode")),
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert set(df.columns) == {"global_track_id", "barcode"}
    print("✓ fieldnames OK")
finally:
    gw.close()


,global_track_id,barcode
0,10,A001
1,20,B002
2,30,C003
3,40,D004


✓ fieldnames OK


In [7]:
# column_names: positional rename after semantic rename
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    df_params=DataFrameParams(
        fieldnames=("global_track_id", "barcode"),
        column_names=["track", "code"],
    ),
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert list(df.columns) == ["track", "code"]
    print("✓ column_names OK")
finally:
    gw.close()


,track,code
0,10,A001
1,20,B002
2,30,C003
3,40,D004


✓ column_names OK


In [8]:
# index_col: promote a column to the DataFrame index
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    df_params=DataFrameParams(index_col="global_track_id"),
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert df.index.name == "global_track_id"
    print("✓ index_col OK")
finally:
    gw.close()


,id,product_type_id,barcode,event_dt
global_track_id,,,,
10,1,1,A001,2024-01-10 00:00:00+00:00
20,2,1,B002,2024-02-15 00:00:00+00:00
30,3,2,C003,2024-03-20 00:00:00+00:00
40,4,2,D004,2024-04-05 00:00:00+00:00


✓ index_col OK


## 7. DataFrameOptions: sort, dedup

In [9]:
# sort_field
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    df_options=DataFrameOptions(sort_field="global_track_id"),
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert list(df["global_track_id"]) == sorted(df["global_track_id"].tolist())
    print("✓ sort_field OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00
2,3,2,30,C003,2024-03-20 00:00:00+00:00
3,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ sort_field OK


In [10]:
# duplicate_expr + keep
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    sticky_filters={"product_type_id": 1},
    df_options=DataFrameOptions(duplicate_expr=["product_type_id"], duplicate_keep="first"),
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert len(df) == 1, f"Expected 1 after dedup, got {len(df)}"
    print("✓ dedup OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00


✓ dedup OK


## 8. sticky_filters & runtime kwargs

Sticky filters are always applied; runtime kwargs are merged on top.

In [11]:
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    sticky_filters={"product_type_id": 1},
)
try:
    df_all = gw.load(as_pandas=True)          # sticky only → 2 rows
    df_one = gw.load(global_track_id=10, as_pandas=True)  # sticky + runtime → 1 row
    display(df_all)
    display(df_one)
    assert len(df_all) == 2
    assert len(df_one) == 1
    print("✓ sticky_filters + runtime kwargs OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00


✓ sticky_filters + runtime kwargs OK


## 9. exclude=True

Wraps the entire filter set in  — returns rows that do NOT match.

In [12]:
gw = DataGateway(
    legacy_cfg(), table="products", field_map=FIELD_MAP,
    sticky_filters={"product_type_id": 1},
    exclude=True,
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert len(df) == 2, f"Expected 2 excluded rows, got {len(df)}"
    assert set(df["product_type_id"].tolist()) == {2}
    print("✓ exclude OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,3,2,30,C003,2024-03-20 00:00:00+00:00
1,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ exclude OK


## 10. persist=True

For Dask DataFrames this pins the graph on workers. Works silently with .

In [13]:
gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    df = gw.load(persist=True, as_pandas=True)
    display(df)
    assert len(df) == 4
    print("✓ persist=True OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00
2,3,2,30,C003,2024-03-20 00:00:00+00:00
3,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ persist=True OK


## 11. timeout= (async)

Wraps the load coroutine in ; raises  if exceeded.

In [14]:
import asyncio

gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    df = await gw.aload(timeout=30, as_pandas=True)
    display(df)
    assert len(df) == 4
    print("✓ timeout (not exceeded) OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00
2,3,2,30,C003,2024-03-20 00:00:00+00:00
3,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ timeout (not exceeded) OK


## 12. load_period / aload_period

Date-range shorthand. Translates semantic  through  automatically.

In [15]:
gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    # single date
    df_single = gw.load_period("event_dt", "2024-02-15", "2024-02-15", as_pandas=True)
    display(df_single)
    assert len(df_single) == 1

    # range
    df_range = gw.load_period("event_dt", "2024-01-01", "2024-02-28", as_pandas=True)
    display(df_range)
    assert len(df_range) == 2

    # async variant
    df_async = await gw.aload_period("event_dt", "2024-01-01", "2024-12-31", as_pandas=True)
    display(df_async)
    assert len(df_async) == 4

    print("✓ load_period / aload_period OK")
finally:
    gw.close()


,id,product_type_id,global_track_id,barcode,event_dt
0,2,1,20,B002,2024-02-15 00:00:00+00:00


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
1,2,1,20,B002,2024-02-15 00:00:00+00:00
2,3,2,30,C003,2024-03-20 00:00:00+00:00
3,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ load_period / aload_period OK


## 13. _has_any_rows()

In [16]:
full_df = pd.DataFrame({"a": [1, 2, 3]})
empty_df = pd.DataFrame({"a": []})
no_cols = pd.DataFrame()

assert DataGateway._has_any_rows(full_df)  is True
assert DataGateway._has_any_rows(empty_df) is False
assert DataGateway._has_any_rows(no_cols)  is False

print("✓ _has_any_rows OK")


✓ _has_any_rows OK


## 14. Chunked  List Splitting (async)

When a  list exceeds , the gateway fires concurrent sub-queries and concatenates the results.

In [17]:
gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    # in_chunk_size=1 forces one query per ID — all 4 rows must come back
    df = await gw.aload(
        global_track_id__in=[10, 20, 30, 40],
        in_chunk_size=1,
        as_pandas=False,
    )
    pdf = df.compute()
    display(pdf)
    assert len(pdf) == 4
    assert set(pdf["global_track_id"].tolist()) == {10, 20, 30, 40}
    print("✓ chunked __in OK")
finally:
    gw.close()


/Users/lvalverdeb/TeamDev/repo-split/boti-data/notebooks/../src/boti_data/db/partitioned_loader.py:36: UserWarning: worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
  self._worker_config = WorkerSqlConfig.from_database_config(config)


,id,product_type_id,global_track_id,barcode,event_dt
0,1,1,10,A001,2024-01-10 00:00:00+00:00
0,2,1,20,B002,2024-02-15 00:00:00+00:00
0,3,2,30,C003,2024-03-20 00:00:00+00:00
0,4,2,40,D004,2024-04-05 00:00:00+00:00


✓ chunked __in OK


## 15. from_config() Classmethod

Accepts the legacy DfHelper-style dict with , , , .

In [18]:
gw = DataGateway.from_config(
    {
        "backend": "sqlalchemy",
        "connection_url": LEGACY_DSN,
        "table": "products",
        "field_map": FIELD_MAP,
        "sticky_filters": {"product_type_id": 2},
        "df_params": {"fieldnames": ("global_track_id", "barcode")},
        "df_options": {"sort_field": "global_track_id"},
    },
    query_only=False,
)
try:
    df = gw.load(as_pandas=True)
    display(df)
    assert set(df.columns) == {"global_track_id", "barcode"}
    assert list(df["global_track_id"]) == sorted(df["global_track_id"].tolist())
    assert len(df) == 2
    print("✓ from_config() OK")
finally:
    gw.close()


,global_track_id,barcode
0,30,C003
1,40,D004


✓ from_config() OK


## 16. Lazy Multi-DataFrame Join

You can keep the entire join graph lazy by merging the gateway result with two or more Dask DataFrames, then materialize only the final result.

In [19]:
gw = DataGateway(legacy_cfg(), table="products", field_map=FIELD_MAP)
try:
    products = gw.load()  # lazy dd.DataFrame

    type_lookup = dd.from_pandas(
        pd.DataFrame(
            {
                "product_type_id": pd.Series([1, 2], dtype="Int64"),
                "product_family": ["core", "plus"],
            }
        ),
        npartitions=1,
    )
    track_status = dd.from_pandas(
        pd.DataFrame(
            {
                "global_track_id": pd.Series([10, 20, 30, 40], dtype="Int64"),
                "status": ["new", "active", "retired", "archived"],
            }
        ),
        npartitions=2,
    )

    joined = products.merge(type_lookup, on="product_type_id", how="left")
    joined = joined.merge(track_status, on="global_track_id", how="left")

    assert isinstance(joined, dd.DataFrame)

    result = joined.compute().sort_values("global_track_id").reset_index(drop=True)
    display(result)
    assert list(result["product_family"]) == ["core", "core", "plus", "plus"]
    assert list(result["status"]) == ["new", "active", "retired", "archived"]
    assert result[["global_track_id", "barcode"]].to_dict("records") == [
        {"global_track_id": 10, "barcode": "A001"},
        {"global_track_id": 20, "barcode": "B002"},
        {"global_track_id": 30, "barcode": "C003"},
        {"global_track_id": 40, "barcode": "D004"},
    ]
    print("✓ lazy multi-join OK")
finally:
    gw.close()


/Users/lvalverdeb/TeamDev/repo-split/boti-data/notebooks/../src/boti_data/db/partitioned_loader.py:36: UserWarning: worker_connection_env_var is not set; falling back to raw DSN. Credentials may be visible in scheduler logs.
  self._worker_config = WorkerSqlConfig.from_database_config(config)


,id,product_type_id,global_track_id,barcode,event_dt,product_family,status
0,1,1,10,A001,2024-01-10 00:00:00+00:00,core,new
1,2,1,20,B002,2024-02-15 00:00:00+00:00,core,active
2,3,2,30,C003,2024-03-20 00:00:00+00:00,plus,retired
3,4,2,40,D004,2024-04-05 00:00:00+00:00,plus,archived


✓ lazy multi-join OK


## Cleanup

In [20]:
import shutil
shutil.rmtree(_TMP_DIR, ignore_errors=True)
print(f"Temp directory {_TMP_DIR} removed.")


Temp directory /var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_nb_f00sr1oo removed.
